# 📊 MSFT Comments — Merge Semanal

Este notebook unifica todas as planilhas `Comment Table [Rede Social] [Mês] [Dia-Dia]` no banco de dados principal `MSFT_Comments`.

**Regras de merge:**
- Duplicatas detectadas por: `Created Time` + `Message` + `SenderScreenName` + `SocialNetwork`
- Se a linha já existe: preenche apenas colunas vazias (sem sobrescrever)
- Coluna `Week` calculada automaticamente no formato `AAAAMM_W#` (ex: `202604_W2`)
- Planilhas processadas são movidas para a subpasta `_processadas`

---
**Como usar toda semana:**
1. Coloque as novas planilhas `Comment Table [...]` na pasta do Drive
2. Execute todas as células (`Runtime > Run all`)
3. Pronto! O `MSFT_Comments` será atualizado automaticamente

## 1. Instalação e Autenticação

In [1]:
!pip install gspread gspread-dataframe --quiet

In [2]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print('✅ Autenticado com sucesso!')

✅ Autenticado com sucesso!


## 2. Configuração — Edite Aqui

In [3]:
# ============================================================
# ⚙️  CONFIGURAÇÕES — edite conforme necessário
# ============================================================

# ID da pasta no Google Drive que contém todas as planilhas
FOLDER_ID = "1DLnJXG72ZpLL5PMDpzzF1bp11jTGwTe6"

# Nome exato da planilha banco de dados
MSFT_SHEET_NAME = "MSFT_Comments"

# Colunas usadas para detectar duplicatas
DUPLICATE_KEY_COLS = ["CreatedTime", "Message", "SenderScreenName", "SocialNetwork"]

# Prefixos aceitos para as planilhas de comentários semanais
# Aceita tanto 'Comment Table' quanto 'Comments Table'
SOURCE_PREFIXES = ["Comment Table", "Comments Table"]

# Nome da subpasta para planilhas processadas (criada automaticamente)
PROCESSED_FOLDER_NAME = "_processadas"

# ============================================================
print('⚙️  Configurações carregadas!')

⚙️  Configurações carregadas!


## 3. Funções Auxiliares

In [4]:
import pandas as pd
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from googleapiclient.discovery import build

drive_service = build('drive', 'v3', credentials=creds)


def listar_arquivos_na_pasta(folder_id):
    """Lista Google Sheets e arquivos Excel (.xlsx) em uma pasta."""
    # Aceita Google Sheets nativos E arquivos .xlsx
    mime_types = [
        "application/vnd.google-apps.spreadsheet",
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    ]
    mime_filter = " or ".join([f"mimeType='{m}'" for m in mime_types])
    query = f"'{folder_id}' in parents and ({mime_filter}) and trashed=false"
    results = drive_service.files().list(q=query, fields='files(id, name, mimeType)').execute()
    return results.get('files', [])


def criar_ou_buscar_subpasta(parent_folder_id, nome_subpasta):
    """Cria a subpasta _processadas se nao existir, retorna o ID."""
    query = (
        f"'{parent_folder_id}' in parents "
        f"and name='{nome_subpasta}' "
        "and mimeType='application/vnd.google-apps.folder' "
        "and trashed=false"
    )
    results = drive_service.files().list(q=query, fields='files(id, name)').execute()
    arquivos = results.get('files', [])
    if arquivos:
        print(f'📁 Subpasta "{nome_subpasta}" encontrada.')
        return arquivos[0]['id']
    else:
        print(f'📁 Criando subpasta "{nome_subpasta}"...')
        metadata = {
            'name': nome_subpasta,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_folder_id]
        }
        folder = drive_service.files().create(body=metadata, fields='id').execute()
        print('✅ Subpasta criada!')
        return folder['id']


def mover_arquivo(file_id, origem_folder_id, destino_folder_id):
    """Move um arquivo de uma pasta para outra no Drive."""
    result = drive_service.files().update(
        fileId=file_id,
        addParents=destino_folder_id,
        removeParents=origem_folder_id,
        fields='id, parents'
    ).execute()
    # Verificar se a mudança de pasta foi bem-sucedida
    if destino_folder_id not in result.get('parents', []):
        raise Exception(f"Arquivo não foi movido. Parents atuais: {result.get('parents')}")


def ler_sheet_como_df(spreadsheet_id, mime_type=None):
    """Le a primeira aba de um Google Sheet ou xlsx como DataFrame."""
    xlsx_mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    if mime_type == xlsx_mime:
        # Exportar xlsx como bytes e ler com pandas
        import io
        content = drive_service.files().get_media(fileId=spreadsheet_id).execute()
        df = pd.read_excel(io.BytesIO(content), dtype=str)
    else:
        sh = gc.open_by_key(spreadsheet_id)
        ws = sh.get_worksheet(0)
        df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str)
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df


def criar_chave_duplicata(df, colunas_chave):
    """Cria chave composta para deteccao de duplicatas."""
    colunas_presentes = [c for c in colunas_chave if c in df.columns]
    return df[colunas_presentes].fillna('').astype(str).agg('||'.join, axis=1)


def calcular_week(created_time_str):
    """
    Calcula coluna Week a partir do Created Time.
    Equivalente a: ANO & TEXTO(MES;'00') & '_W' & INT((DIA-1)/7)+1
    Resultado: ex. 202604_W2
    """
    if pd.isna(created_time_str) or str(created_time_str).strip() == '':
        return ''
    try:
        dt = pd.to_datetime(created_time_str, infer_datetime_format=True)
        ano    = dt.year
        mes    = str(dt.month).zfill(2)
        semana = int((dt.day - 1) / 7) + 1
        return f'{ano}{mes}_W{semana}'
    except Exception:
        return ''


print('✅ Funcoes auxiliares carregadas!')

✅ Funcoes auxiliares carregadas!


## 4. Merge — Execução Principal

In [5]:
print('='*60)
print('🚀 Iniciando merge semanal...')
print('='*60)

# ── 1. Listar arquivos na pasta ──────────────────────────────
todos_arquivos = listar_arquivos_na_pasta(FOLDER_ID)

msft_file  = None
source_files = []

for arquivo in todos_arquivos:
    if arquivo['name'] == MSFT_SHEET_NAME:
        msft_file = arquivo
    elif any(arquivo['name'].startswith(p) for p in SOURCE_PREFIXES):
        source_files.append(arquivo)

if not msft_file:
    raise ValueError(f'❌ Planilha "{MSFT_SHEET_NAME}" nao encontrada na pasta!')

print(f'\n📋 Banco de dados: {MSFT_SHEET_NAME}')
print(f'📂 Planilhas de origem encontradas: {len(source_files)}')
for f in source_files:
    print(f'   - {f["name"]}')

if not source_files:
    print('\n⚠️  Nenhuma planilha "Comment Table" encontrada. Nada a fazer!')
else:
    # ── 2. Ler MSFT_Comments ─────────────────────────────────
    print('\n📖 Lendo MSFT_Comments...')
    df_msft = ler_sheet_como_df(msft_file['id'])
    print(f'   Linhas atuais no banco: {len(df_msft)}')

    # ── 2b. Backfill Week em linhas existentes vazias ────────
    if 'Week' in df_msft.columns and 'CreatedTime' in df_msft.columns:
        mask_vazio = df_msft['Week'].isna() | (df_msft['Week'].astype(str).str.strip() == '')
        df_msft.loc[mask_vazio, 'Week'] = df_msft.loc[mask_vazio, 'CreatedTime'].apply(calcular_week)
        if mask_vazio.sum() > 0:
            print(f'   ↩️  Week preenchida em {mask_vazio.sum()} linhas existentes')

    # ── Corrigir Week com hífen para underscore ───────────────────
if 'Week' in df_msft.columns:
    df_msft['Week'] = df_msft['Week'].astype(str).str.replace(r'(\d{6})-W', r'\1_W', regex=True)
    print('✅ Padronização de Week concluída (hífen → underscore)')


    # ── 3. Criar subpasta _processadas ───────────────────────
    processed_folder_id = criar_ou_buscar_subpasta(FOLDER_ID, PROCESSED_FOLDER_NAME)

    # ── 4. Processar cada planilha de origem ─────────────────
    total_novas      = 0
    total_atualizadas = 0
    total_ignoradas  = 0

    for arquivo in source_files:
        print(f'\n📄 Processando: {arquivo["name"]}')
        df_origem = ler_sheet_como_df(arquivo['id'], mime_type=arquivo.get('mimeType'))
        print(f'   Linhas encontradas: {len(df_origem)}')

        # Garantir que todas as colunas da origem existam no banco
        for col in df_origem.columns:
            if col not in df_msft.columns:
                df_msft[col] = pd.NA

        # Garantir coluna Week no banco
        if 'Week' not in df_msft.columns:
            df_msft['Week'] = ''

        chaves_msft = criar_chave_duplicata(df_msft, DUPLICATE_KEY_COLS)
        novas_linhas = []

        for _, row_nova in df_origem.iterrows():
            chave_nova = '||'.join(
                str(row_nova.get(c, '')) if pd.notna(row_nova.get(c, '')) else ''
                for c in DUPLICATE_KEY_COLS if c in df_origem.columns
            )

            idx_existente = chaves_msft[chaves_msft == chave_nova].index

            if len(idx_existente) == 0:
                # Linha nova: calcular Week e adicionar
                row_nova = row_nova.copy()
                row_nova['Week'] = calcular_week(row_nova.get('CreatedTime', ''))
                novas_linhas.append(row_nova)
                total_novas += 1
            else:
                # Duplicata: preencher apenas colunas vazias
                idx = idx_existente[0]
                atualizado = False
                for col in df_origem.columns:
                    val_existente = df_msft.at[idx, col] if col in df_msft.columns else pd.NA
                    val_novo = row_nova.get(col, pd.NA)
                    if (pd.isna(val_existente) or str(val_existente).strip() == '') and \
                       not (pd.isna(val_novo) or str(val_novo).strip() == ''):
                        df_msft.at[idx, col] = val_novo
                        atualizado = True
                # Preencher Week se vazio
                week_atual = df_msft.at[idx, 'Week']
                if pd.isna(week_atual) or str(week_atual).strip() == '':
                    df_msft.at[idx, 'Week'] = calcular_week(
                        df_msft.at[idx, 'CreatedTime'] if 'CreatedTime' in df_msft.columns else ''
                    )
                    atualizado = True
                if atualizado:
                    total_atualizadas += 1
                else:
                    total_ignoradas += 1

        # Adicionar novas linhas
        if novas_linhas:
            df_novas = pd.DataFrame(novas_linhas).reindex(columns=df_msft.columns)
            df_msft  = pd.concat([df_msft, df_novas], ignore_index=True)

        print(f'   ✅ Novas: {len(novas_linhas)} | Atualizadas: {total_atualizadas} | Ignoradas: {total_ignoradas}')

# Mover para _processadas
        try:
            mover_arquivo(arquivo['id'], FOLDER_ID, processed_folder_id)
            print(f'   📦 Movida para "{PROCESSED_FOLDER_NAME}"')
        except Exception as e:
            print(f'   ⚠️  Erro ao mover "{arquivo["name"]}": {e}')

    # ── 5. Salvar MSFT_Comments atualizado ───────────────────
    print('\n💾 Salvando MSFT_Comments...')
    sh_msft  = gc.open_by_key(msft_file['id'])
    ws_msft  = sh_msft.get_worksheet(0)
    ws_msft.clear()
    set_with_dataframe(ws_msft, df_msft)

    print('\n' + '='*60)
    print('🎉 Merge concluido!')
    print(f'   Total de linhas no banco agora : {len(df_msft)}')
    print(f'   Novas linhas adicionadas       : {total_novas}')
    print(f'   Linhas atualizadas (col vazias): {total_atualizadas}')
    print(f'   Linhas duplicadas ignoradas    : {total_ignoradas}')
    print('='*60)

🚀 Iniciando merge semanal...

📋 Banco de dados: MSFT_Comments
📂 Planilhas de origem encontradas: 0

⚠️  Nenhuma planilha "Comment Table" encontrada. Nada a fazer!


NameError: name 'df_msft' is not defined